# Perceptrón Multicapa (1 Capa Oculta) — Clasificación binaria de género en Twitter

**Taller 1 — Deep Learning (2026-10)**

**Modelo:** Red Neuronal Feed-Forward (MLP) con una capa oculta.

**Objetivo:** Clasificar usuarios de Twitter como `male (0)` o `female (1)`.

Los datos ya fueron preprocesados en `Taller1DL.ipynb` (secciones 1-5) y se cargan desde archivos `.npy`.

---

## 1. Carga de datos preprocesados

Los archivos `.npy` contienen las matrices ya:
- Imputadas (nulos tratados)
- Escaladas (StandardScaler para numéricas)
- Codificadas (OneHotEncoder para categóricas)
- Particionadas (60% train, 20% val, 20% test)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar matrices preprocesadas generadas en Taller1DL.ipynb
X_train = np.load("X_train.npy")
X_val   = np.load("X_val.npy")
X_test  = np.load("X_test.npy")
y_train = np.load("y_train.npy")
y_val   = np.load("y_val.npy")
y_test  = np.load("y_test.npy")

print("="*50)
print("  DIMENSIONES DE LOS DATOS")
print("="*50)
print(f"  Train: X={X_train.shape}, y={y_train.shape}")
print(f"  Val:   X={X_val.shape},   y={y_val.shape}")
print(f"  Test:  X={X_test.shape},  y={y_test.shape}")
print(f"\n  Número de features: {X_train.shape[1]}")
print(f"  Clases únicas: {np.unique(y_train)}  (0=male, 1=female)")

numero_entradas = X_train.shape[1]

## 2. ¿Qué es el Perceptrón Multicapa (MLP) con 1 Capa Oculta?

El **Perceptrón Multicapa (MLP)** es una red neuronal artificial feed-forward. Supera las limitaciones del perceptrón unicapa, permitiendo aprender patrones y fronteras de decisión **no lineales**.

### Arquitectura:
```
Capa de Entrada (x)    Capa Oculta (h)      Capa de Salida (ŷ)

x₁ ───┐                ┌─── h₁ ───┐
x₂ ───┼─── W⁽¹⁾ ───────┼─── h₂ ───┼─── W⁽²⁾ ──→ ŷ ∈ {0, 1}
 ⋮    │                │    ⋮     │
xₙ ───┘                └─── hₘ ───┘
```

Para este punto el taller establece que **el número de neuronas en la capa oculta ($m$) debe ser igual al número de entradas ($n$)**.
Es decir, $m = n = 4323$ (basado en el shape de las features preprocesadas).

### Componentes:
- **Capa de Entrada:** Recibe las $n$ features originales (entradas $x_i$).
- **Capa Oculta (Hidden layer):** Contiene $m$ neuronas. Aplica una transformación lineal a las entradas, seguida de una función de activación no lineal (**ReLU**).
- **Capa de Salida:** Contiene 1 sola neurona (por ser clasificación binaria). Procesa la salida de la capa oculta y le aplica una función de activación (**Sigmoide**).
- **Pesos y Sesgos:** Matrices de pesos $W^{(1)}, W^{(2)}$ y sesgos $b^{(1)}, b^{(2)}$ que se ajustan durante el entrenamiento.

### Funciones de Activación:

**1. Capa Oculta (ReLU - Rectified Linear Unit):**
$$h_j = \max(0, \sum_{i=1}^{n} W^{(1)}_{ji} x_i + b^{(1)}_j)$$
Introduce la no-linealidad en el modelo, permitiendo aprender representaciones complejas. Es computacionalmente eficiente y mitiga el problema del desvanecimiento del gradiente.

**2. Capa de Salida (Sigmoide):**
$$\hat{y} = \sigma(\sum_{j=1}^{m} W^{(2)}_{j} h_j + b^{(2)}) = \frac{1}{1 + e^{-z}}$$
Mapea el resultado a un valor continuo entre $0$ y $1$. En problemas bi-clase (female vs male), el umbral por defecto es $0.5$.

### Regla de Aprendizaje (Backpropagation):
A diferencia del perceptrón simple (que actualiza según el error regla delta), el MLP utiliza el algoritmo de **retropropagación (backpropagation)** en conjunto con optimizadores (como SGD o Adam) para minimizar una función de pérdida (en nuestro caso, **Binary Crossentropy**). Los errores se propagan desde la salida hacia las capaz ocultas:
$$W \leftarrow W - \eta \frac{\partial L}{\partial W}$$

## 3. Construcción y Entrenamiento del Modelo MLP
A diferencia del Perceptrón de Scikit-Learn, aquí utilizaremos **Keras (TensorFlow)** para construir la red neuronal multicapa.

### Hiperparámetros:
| Parámetro | Valor | Descripción |
|---|---|---|
| Optimizador | Adam | Método de descenso de gradiente estocástico eficiente |
| Función de Pérdida | Binary Crossentropy | Ideal para clasificación binaria |
| Épocas | 100 | Ciclos completos de entrenamiento |
| Batch Size | 32 | Número de muestras procesadas antes de actualizar los pesos |

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

# Definición de la arquitectura del modelo
model = Sequential([
    Input(shape=(numero_entradas,)),
    Dense(numero_entradas, activation='relu', name='capa_oculta'),
    Dense(1, activation='sigmoid', name='capa_salida')
])

# Compilación
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("="*50)
print("  ARQUITECTURA DEL MODELO")
print("="*50)
model.summary()

In [ ]:
# Entrenamiento
print("\nIniciando entrenamiento...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    verbose=1
)

### Evolución del Entrenamiento (Pérdida y Precisión)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Gráfica de Loss
ax[0].plot(history.history['loss'], label='Train Loss')
ax[0].plot(history.history['val_loss'], label='Validation Loss')
ax[0].set_title('Evolución de la Función de Pérdida (Loss)')
ax[0].set_xlabel('Época')
ax[0].set_ylabel('Loss (Binary Crossentropy)')
ax[0].legend()
ax[0].grid(True, linestyle='--', alpha=0.6)

# Gráfica de Accuracy
ax[1].plot(history.history['accuracy'], label='Train Accuracy')
ax[1].plot(history.history['val_accuracy'], label='Validation Accuracy')
ax[1].set_title('Evolución de la Precisión (Accuracy)')
ax[1].set_xlabel('Época')
ax[1].set_ylabel('Accuracy')
ax[1].legend()
ax[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## 4. Predicciones

In [ ]:
# Predicciones (Probabilidades)
y_train_prob = model.predict(X_train)
y_val_prob   = model.predict(X_val)
y_test_prob  = model.predict(X_test)

# Conversión a clases (umbral 0.5)
y_train_pred = (y_train_prob > 0.5).astype(int).flatten()
y_val_pred   = (y_val_prob > 0.5).astype(int).flatten()
y_test_pred  = (y_test_prob > 0.5).astype(int).flatten()

# Verificación rápida
print("="*50)
print("  MUESTRA DE PREDICCIONES (Test, primeras 20)")
print("="*50)
print(f"  Real:      {y_test[:20]}")
print(f"  Predicho:  {y_test_pred[:20]}")
coincidencias = np.sum(y_test[:20] == y_test_pred[:20])
print(f"  Aciertos:  {coincidencias}/20")

## 5. Evaluación y Métricas

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

def evaluar(y_true, y_pred, nombre):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)

    print(f"\n{'='*55}")
    print(f"  {nombre}")
    print(f"{'='*55}")
    print(f"  Accuracy:  {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1 Score:  {f1:.4f}\n")
    print(classification_report(y_true, y_pred, target_names=["male (0)", "female (1)"]))

    return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1}

m_train = evaluar(y_train, y_train_pred, "TRAIN")
m_val   = evaluar(y_val, y_val_pred, "VALIDACIÓN")
m_test  = evaluar(y_test, y_test_pred, "TEST")

resumen = pd.DataFrame({
    "Train": m_train,
    "Validación": m_val,
    "Test": m_test
}).T

print("\nMLP 1 Capa Oculta — Resumen de métricas")
print("="*55)
display(resumen.round(4))

## 6. Matriz de Confusión (Conjunto de Prueba)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_test_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["male (0)", "female (1)"]
)

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap='Blues', ax=ax, values_format='d')
plt.title('Matriz de Confusión - MLP 1 Capa Oculta (Test)')
plt.grid(False)
plt.show()